# Experiment 0.1.2 — Regularized General Comparison
Analysis-only notebook. Training and evaluation are performed by the Slurm array; this notebook reads finalized artifacts only.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

def find_repo_root(start=Path.cwd()):
    for candidate in (start, *start.parents):
        if (candidate / 'scripts').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise FileNotFoundError('Could not locate writingRing repository root')

REPO_ROOT = find_repo_root()
ART = REPO_ROOT / 'notebooks' / 'artifacts' / 'experiment_0_1_2_regularized_general_comparison' / 'regularized_general_comparison_v1'
manifest = json.loads((ART / 'manifest.json').read_text())
manifest


In [ ]:
runs = pd.read_csv(ART / 'runs.csv')
summary = pd.read_csv(ART / 'summary.csv')
comparison = pd.read_csv(ART / 'comparison_summary.csv')
paired = pd.read_csv(ART / 'paired_regularization_effects.csv')
objective_interactions = pd.read_csv(ART / 'regularization_objective_interactions.csv')
capacity_interactions = pd.read_csv(ART / 'regularization_capacity_interactions.csv')
architecture_interactions = pd.read_csv(ART / 'regularization_architecture_interactions.csv')
display(summary.sort_values('test_ba_mean', ascending=False).head(30))
display(comparison.sort_values('mean_test_ba', ascending=False).head(30))


## Paired regularization effect
Positive `delta_test_ba_reg_on_minus_off` means the hidden P2+A1 condition improved test BA for the same architecture/objective/capacity/seed.


In [ ]:
paired_summary = (
    paired.groupby(['architecture', 'objective', 'variant'])['delta_test_ba_reg_on_minus_off']
    .agg(['mean', 'std', 'count'])
    .reset_index()
    .sort_values('mean', ascending=False)
)
display(paired_summary)
display(paired.groupby(['objective', 'variant'])['delta_test_ba_reg_on_minus_off'].agg(['mean', 'std', 'count']))


In [ ]:
plot_df = paired_summary.copy()
plot_df['condition'] = plot_df['architecture'] + ' | ' + plot_df['objective'] + ' | ' + plot_df['variant']
ax = plot_df.plot.bar(x='condition', y='mean', yerr='std', legend=False, figsize=(12, 5))
ax.axhline(0.0, linewidth=1)
ax.set_ylabel('Mean paired Δ test BA (reg on - off)')
ax.set_xlabel('')
plt.xticks(rotation=70, ha='right')
plt.tight_layout()


## Interaction checks


In [ ]:
display(objective_interactions.groupby(['architecture', 'variant'])['interaction_reg_delta_timestep_minus_whole_count'].agg(['mean', 'std', 'count']))
display(capacity_interactions.groupby(['architecture', 'objective'])['interaction_reg_delta_multi_h_minus_binary'].agg(['mean', 'std', 'count']))
display(architecture_interactions.groupby(['comparison', 'objective', 'variant'])['interaction_reg_delta_candidate_minus_baseline'].agg(['mean', 'std', 'count']))


## Reg-on neuron-health diagnostics
The frozen Exp0.1 reg-off rows do not contain the new dead/cap diagnostics, so these columns are interpreted only within reg-on runs. The paired last-hidden event-rate delta remains available for reg-on vs reg-off.


In [ ]:
reg_on = runs[(runs['family'] == 'direct_snn') & (runs['regularization'] == 'on')]
diag_cols = [
    'test_mean_dead_neuron_fraction',
    'test_mean_active_step_fraction',
    'test_mean_fraction_at_cap',
    'test_last_hidden_dead_neuron_fraction',
    'test_last_hidden_active_step_fraction',
    'test_last_hidden_fraction_at_cap',
]
display(reg_on.groupby(['architecture', 'objective', 'variant'])[diag_cols].mean())
display(paired.groupby(['architecture', 'objective', 'variant'])['delta_last_hidden_events_per_neuron_second'].agg(['mean', 'std', 'count']))
